In [1]:
pip install git+https://github.com/huggingface/parler-tts.git

  Cloning https://github.com/huggingface/parler-tts.git to C:\Users\akash\AppData\Local\Temp\pip-req-build-e3anjzrl
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/descriptinc/audiotools to C:\Users\akash\AppData\Local\Temp\pip-install-lrffqdwq\descript-audiotools_c9977ff8d6bc43a98d2cddebf1de54db
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to buil

  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git 'C:\Users\akash\AppData\Local\Temp\pip-req-build-e3anjzrl'
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools 'C:\Users\akash\AppData\Local\Temp\pip-install-lrffqdwq\descript-audiotools_c9977ff8d6bc43a98d2cddebf1de54db'


In [2]:
from huggingface_hub import login
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

# Only needed if you are not already logged in
# login()

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained(
    "ai4bharat/indic-parler-tts",
    token=True
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indic-parler-tts",
    token=True
)

description_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path,
    token=True
)

print("Model and both tokenizers loaded successfully.")
print("Description tokenizer source:", model.config.text_encoder._name_or_path)


c:\Users\akash\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Flash attention 2 is not installed
Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token

Model and both tokenizers loaded successfully.
Description tokenizer source: google/flan-t5-large


In [3]:
import os
import sys
import uuid
import numpy as np
import soundfile as sf
import torch
import gradio as gr

from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

# =========================================================
# 1) Config
# =========================================================
MODEL_ID = "ai4bharat/indic-parler-tts"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE.startswith("cuda") else torch.float32

OUTPUT_DIR = "tts_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IS_COLAB = "google.colab" in sys.modules

# =========================================================
# 2) Sample texts
# =========================================================
SAMPLE_TEXTS = {
    "english": "Today is a beautiful day, and I am very happy to speak with you.",
    "hindi": "आज का दिन बहुत सुंदर है और आपसे बात करके मुझे बहुत खुशी हो रही है।",
    "kannada": "ಇಂದು ಬಹಳ ಸುಂದರವಾದ ದಿನವಾಗಿದೆ ಮತ್ತು ನಿಮ್ಮೊಂದಿಗೆ ಮಾತನಾಡುವುದು ನನಗೆ ತುಂಬ ಸಂತೋಷವಾಗಿದೆ.",
    "telugu": "ఈ రోజు చాలా అందమైన రోజు, మీతో మాట్లాడటం నాకు చాలా ఆనందంగా ఉంది.",
    "marathi": "आजचा दिवस खूप सुंदर आहे आणि तुमच्याशी बोलताना मला खूप आनंद होत आहे.",
    "gujarati": "આજનો દિવસ ખૂબ સુંદર છે અને તમારી સાથે વાત કરીને મને ખૂબ આનંદ થાય છે."
}

# =========================================================
# 3) Description map: language + gender only
# =========================================================
DESCRIPTION_MAP = {
    "hindi": {
        "female": "Divya speaks Hindi in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "Rohit speaks Hindi in a soft, warm, natural conversational tone. Very clear audio."
    },
    "english": {
        "female": "Mary speaks English in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "Thoma speaks English in a soft, warm, natural conversational tone. Very clear audio."
    },
    "marathi": {
        "female": "A female speaker speaks Marathi in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "A male speaker speaks Marathi in a soft, warm, natural conversational tone. Very clear audio."
    },
    "telugu": {
        "female": "A female speaker speaks Telugu in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "A male speaker speaks Telugu in a soft, warm, natural conversational tone. Very clear audio."
    },
    "kannada": {
        "female": "A female speaker speaks Kannada in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "A male speaker speaks Kannada in a soft, warm, natural conversational tone. Very clear audio."
    },
    "gujarati": {
        "female": "A female speaker speaks Gujarati in a soft, warm, natural conversational tone. Very clear audio.",
        "male": "A male speaker speaks Gujarati in a soft, warm, natural conversational tone. Very clear audio."
    }
}

def build_description(language: str, gender: str) -> str:
    return DESCRIPTION_MAP[language][gender]

def descriptions_markdown(language: str) -> str:
    return (
        f"### Descriptions for {language.title()}\n\n"
        f"- **Female:** {DESCRIPTION_MAP[language]['female']}\n\n"
        f"- **Male:** {DESCRIPTION_MAP[language]['male']}"
    )

# =========================================================
# 4) Load model + tokenizers
#    If you already loaded these earlier, you can skip this block
#    and keep the rest of the app code.
# =========================================================
print(f"Loading model on: {DEVICE}")

model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=True,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)

description_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path,
    token=True
)

print("Model and tokenizers loaded successfully.")
print("Description tokenizer source:", model.config.text_encoder._name_or_path)

# =========================================================
# 5) Core synthesis
# =========================================================
def save_wav(audio_arr: np.ndarray, sample_rate: int, prefix: str) -> str:
    file_id = uuid.uuid4().hex[:8]
    wav_path = os.path.join(OUTPUT_DIR, f"{prefix}_{file_id}.wav")
    sf.write(wav_path, audio_arr, sample_rate)
    return wav_path

@torch.inference_mode()
def synthesize_core(text: str, language: str, gender: str):
    text = (text or "").strip()
    if not text:
        raise gr.Error("Please enter some text.")

    description = build_description(language, gender)

    description_inputs = description_tokenizer(description, return_tensors="pt").to(DEVICE)
    prompt_inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    generation = model.generate(
        input_ids=description_inputs.input_ids,
        attention_mask=description_inputs.attention_mask,
        prompt_input_ids=prompt_inputs.input_ids,
        prompt_attention_mask=prompt_inputs.attention_mask,
    )

    audio_arr = generation.cpu().numpy().squeeze().astype(np.float32)

    peak = np.max(np.abs(audio_arr)) if audio_arr.size else 0.0
    if peak > 0:
        audio_arr = 0.98 * audio_arr / peak

    wav_path = save_wav(audio_arr, model.config.sampling_rate, f"{language}_{gender}")

    info_md = (
        f"### Playback\n"
        f"- **Language:** {language}\n"
        f"- **Gender:** {gender}\n"
        f"- **Description used:** {description}"
    )

    return wav_path, description, info_md

def generate_single(text: str, language: str, gender: str):
    wav_path, description, info_md = synthesize_core(text, language, gender)
    return wav_path, wav_path, info_md, description

def generate_both(text: str, language: str):
    female_wav, female_desc, _ = synthesize_core(text, language, "female")
    male_wav, male_desc, _ = synthesize_core(text, language, "male")

    compare_md = (
        f"### Voice comparison for {language.title()}\n\n"
        f"**Female description:** {female_desc}\n\n"
        f"**Male description:** {male_desc}"
    )

    return female_wav, male_wav, female_wav, male_wav, compare_md

def load_sample_text(language: str):
    return SAMPLE_TEXTS[language]

def current_description(language: str, gender: str):
    return build_description(language, gender)

# =========================================================
# 6) Gradio UI
# =========================================================
with gr.Blocks(theme=gr.themes.Soft(), title="Multilingual TTS Demo") as demo:
    gr.Markdown(
        """
        # Multilingual Text-to-Speech Demo
        Supports **English, Hindi, Kannada, Telugu, Marathi, and Gujarati**
        using **Indic Parler-TTS**.
        """
    )

    gr.Markdown(f"**Running on:** `{DEVICE}`")

    with gr.Tab("Generate One Voice"):
        with gr.Row():
            language = gr.Dropdown(
                choices=["english", "hindi", "kannada", "telugu", "marathi", "gujarati"],
                value="hindi",
                label="Language"
            )
            gender = gr.Radio(
                choices=["female", "male"],
                value="female",
                label="Voice"
            )

        text = gr.Textbox(
            value=SAMPLE_TEXTS["hindi"],
            lines=5,
            label="Input text"
        )

        with gr.Row():
            load_sample_btn = gr.Button("Load sample text")
            generate_btn = gr.Button("Generate speech", variant="primary")

        description_box = gr.Textbox(
            value=build_description("hindi", "female"),
            label="Current description",
            lines=3,
            interactive=False
        )

        audio_out = gr.Audio(label="Playback", type="filepath")
        wav_download = gr.File(label="Download WAV")
        info_out = gr.Markdown()

    with gr.Tab("Preview Female + Male"):
        compare_language = gr.Dropdown(
            choices=["english", "hindi", "kannada", "telugu", "marathi", "gujarati"],
            value="hindi",
            label="Language"
        )
        compare_text = gr.Textbox(
            value=SAMPLE_TEXTS["hindi"],
            lines=5,
            label="Input text"
        )

        with gr.Row():
            compare_sample_btn = gr.Button("Load sample text")
            compare_btn = gr.Button("Preview both voices", variant="primary")

        with gr.Row():
            female_audio = gr.Audio(label="Female playback", type="filepath")
            male_audio = gr.Audio(label="Male playback", type="filepath")

        with gr.Row():
            female_wav = gr.File(label="Download female WAV")
            male_wav = gr.File(label="Download male WAV")

        compare_info = gr.Markdown()

    with gr.Tab("Descriptions"):
        desc_language = gr.Dropdown(
            choices=["english", "hindi", "kannada", "telugu", "marathi", "gujarati"],
            value="hindi",
            label="Language"
        )
        desc_md = gr.Markdown(value=descriptions_markdown("hindi"))

    # Events
    load_sample_btn.click(
        fn=load_sample_text,
        inputs=language,
        outputs=text
    )

    generate_btn.click(
        fn=generate_single,
        inputs=[text, language, gender],
        outputs=[audio_out, wav_download, info_out, description_box]
    )

    language.change(
        fn=load_sample_text,
        inputs=language,
        outputs=text
    )

    language.change(
        fn=current_description,
        inputs=[language, gender],
        outputs=description_box
    )

    gender.change(
        fn=current_description,
        inputs=[language, gender],
        outputs=description_box
    )

    compare_sample_btn.click(
        fn=load_sample_text,
        inputs=compare_language,
        outputs=compare_text
    )

    compare_btn.click(
        fn=generate_both,
        inputs=[compare_text, compare_language],
        outputs=[female_audio, male_audio, female_wav, male_wav, compare_info]
    )

    compare_language.change(
        fn=load_sample_text,
        inputs=compare_language,
        outputs=compare_text
    )

    desc_language.change(
        fn=descriptions_markdown,
        inputs=desc_language,
        outputs=desc_md
    )

demo.queue()
demo.launch(share=True if IS_COLAB else False, debug=IS_COLAB)

Loading model on: cpu


Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

Config of the audio_encoder: <class 'transformers.models.dac.modelin

Model and tokenizers loaded successfully.
Description tokenizer source: google/flan-t5-large


C:\Users\akash\AppData\Local\Temp\ipykernel_14824\2778606528.py:169: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Multilingual TTS Demo") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
import nbformat

nb = nbformat.read("Untitled76 (1).ipynb", as_version=4)

# remove broken widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, "app_clean.ipynb")

print("Clean notebook created: app_clean.ipynb")

Clean notebook created: app_clean.ipynb
